# Stage 1 - loading, de-duplication and the diversity gradient

Load each dataset's raw CAN frames into the canonical table, measure the duplication that inflates accuracy, strictly de-duplicate to the unique attack signatures, and split into train/test. Every result is shown per dataset so it can go straight into the report.

In [ ]:
import sys
from pathlib import Path
try:
    import adversec
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent)); import adversec
import numpy as np, pandas as pd
from adversec import config
from adversec.contract import FEATURES, LABEL_COLUMN, ID_COLUMN, DATA_COLUMNS
pd.set_option('display.width', 140)

# The two datasets are treated identically: every step below runs the SAME
# code for both. The only dataset-specific code in the project is each
# dataset's loader (adversec/datasets/ciciov.py, road.py).
DATASETS = ['ciciov2024', 'road']

## Step 1 - load the raw canonical table
Each loader converts its raw format (CICIoV decimal CSVs, ROAD candump logs) into the same columns: `ID, DATA_0..DATA_7, true_class`.

In [ ]:
from adversec.datasets.registry import get_dataset
raw = {}
for name in DATASETS:
    raw[name] = get_dataset(name).load()
    print(f'\n=== {name}: {len(raw[name]):,} frames ===')
    display(raw[name].head())

## Step 2 - audit duplication (the accuracy trap)
The fraction of rows that are redundant copies. High duplication is what inflates naive accuracy.

In [ ]:
from adversec.pipeline import audit_duplication
for name in DATASETS:
    a = audit_duplication(raw[name], FEATURES + [LABEL_COLUMN])
    print(f"{name:12s} rows={a['total_rows']:>10,}  unique={a['unique_signatures']:>8,}  duplication={a['duplication_rate_pct']:>7}%")

## Step 3 - strict de-duplication -> the diversity gradient
One row per unique (features + class) signature. The per-class counts are the study's independent variable: how many genuinely distinct attacks each class holds.

In [ ]:
from adversec.pipeline import strict_dedup
strict = {}
for name in DATASETS:
    strict[name] = strict_dedup(raw[name], FEATURES)
    print(f'\n=== {name}: {len(strict[name]):,} unique signatures ===')
    display(strict[name][LABEL_COLUMN].value_counts().rename('unique signatures').to_frame())

## Step 4 - signature-level train/test split
Splitting happens on unique signatures BEFORE any augmentation, so no copy can straddle train and test. Tiny classes get a single-signature test floor.

In [ ]:
from adversec.pipeline import split_train_test
split = {}
for name in DATASETS:
    tr, te = split_train_test(strict[name]); split[name] = (tr, te)
    print(f'\n=== {name}: train {len(tr):,} / test {len(te):,} ===')
    tbl = pd.DataFrame({'train': tr[LABEL_COLUMN].value_counts(), 'test': te[LABEL_COLUMN].value_counts()}).fillna(0).astype(int)
    display(tbl)

## Step 5 - save for the next stage
Written to `datasets/processed/<name>_*.csv` with a consistent per-dataset prefix (equal naming).

In [ ]:
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
for name in DATASETS:
    strict[name].to_csv(config.PROCESSED_DIR / f'{name}_strict.csv', index=False)
    split[name][0].to_csv(config.PROCESSED_DIR / f'{name}_train.csv', index=False)
    split[name][1].to_csv(config.PROCESSED_DIR / f'{name}_test.csv', index=False)
    print('saved', name)